In [ ]:
import os
import sys
from pathlib import Path


ROOT = Path("../")

cuda_vd = 3
CONFIG_PATH = ROOT / "src/configs/ESASRecStageV2___amazon_sports.yaml"
CKPT_PATH = ROOT / "src/lightning_logs/version_5/checkpoints/epoch=80-step=20736.ckpt" 
OUT_PATH = ROOT /"../data/Sports/grid"
OUT_PT_NAME = "new_embeddings_amazon_toys_after_adapter.pt"



ROOT = Path("../")
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
os.chdir(SRC)

if cuda_vd is not None:
    os.environ["CUDA_VISIBLE_DEVICES"] = str(cuda_vd)

import torch
from omegaconf import OmegaConf
from run import create_model, prepare_data
from modules import SeqRec

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
config = OmegaConf.load(CONFIG_PATH)
_, _, _, _, item_count = prepare_data(config)

model = create_model(config, item_count)
seqrec_module = SeqRec(model, **config.seqrec_module)

ckpt = torch.load(CKPT_PATH, map_location="cpu")
seqrec_module.load_state_dict(ckpt["state_dict"], strict=True)

seqrec_module = seqrec_module.to(DEVICE).eval()
model = seqrec_module.model


In [ ]:
item_ids = torch.arange(1, model.item_content_emb.weight.shape[0], device=DEVICE, dtype=torch.long)
chunk = 2048
pieces = []
with torch.no_grad():
    for start in range(0, item_ids.numel(), chunk):
        batch = item_ids[start : start + chunk].unsqueeze(0)  
        out = model.get_embeddings(batch)  
        pieces.append(out.squeeze(0))
all_item_adapter_emb = torch.cat(pieces, dim=0)
print(all_item_adapter_emb.shape)

out_dir = Path(OUT_PATH) 
out_dir.mkdir(parents=True, exist_ok=True)
out_file = out_dir / OUT_PT_NAME
torch.save(all_item_adapter_emb.cpu(), out_file)
print(f"Saved: {out_file}")

all_item_adapter_emb
